# Nemotron Jupyter and Colab Runner

This notebook runs the same training and evaluation pipeline from `app.py` in an interactive workflow.

Supported modes:
- TinyStories (default)
- JSONL chat data (including assistant-only loss masking)

In [ ]:
# Install dependencies.
# Local Jupyter: usually run this once in your virtualenv, then skip.
# Colab: run this cell every new session.

%pip install -q --upgrade pip
%pip install -q --upgrade jax flax optax datasets transformers

In [ ]:
from __future__ import annotations

import jax
import jax.numpy as jnp
import optax
from flax import nnx

from dataclasses import dataclass, field

import argparse
import json
import math
from pathlib import Path
from typing import TYPE_CHECKING

In [ ]:
"""
Minimal Grouped-Query Attention (GQA) in JAX/Flax NNX.

This file implements a simple, educational attention block used by the
Nemotron-style hybrid model:
- Causal self-attention (decoder-only masking)
- Grouped-query attention (more query heads than KV heads)
- Rotary Position Embeddings (RoPE, Su et al. 2021) applied to Q and K
- No dropout, bias-free projections

The implementation prioritizes readability over speed.
"""

# =============================================================================
# Rope Algorithm
# =============================================================================


def _apply_rope(x: jax.Array) -> jax.Array:
    """
    Apply Rotary Position Embeddings (RoPE, Su et al. 2021) to a Q or K tensor.

    RoPE encodes token positions as rotations of head-dimension pairs.
    For dimension pair (2i, 2i+1) at sequence position m, the rotation angle is:
        θ_i = m / 10000^(2i / head_dim)

    Key property: after RoPE, the dot product q·k depends only on the
    RELATIVE position (m − n), not absolute positions. This lets the model
    generalize to positions seen at training time.

    RoPE adds no learnable parameters — it is a deterministic function of position.

    Args:
        x: shape (batch, seqlen, num_heads, head_dim) — Q or K tensor
    Returns:
        shape (batch, seqlen, num_heads, head_dim) — position-encoded
    """
    batch, seqlen, num_heads, head_dim = x.shape
    if head_dim % 2 != 0:
        raise ValueError("RoPE requires an even head_dim")
    half = head_dim // 2

    # Frequency for each dimension pair: θ_i = 1 / 10000^(2i / head_dim)
    freqs = 1.0 / (10000.0 ** (jnp.arange(half, dtype=jnp.float32) * 2 / head_dim))

    # Position indices 0, 1, ..., seqlen-1
    positions = jnp.arange(seqlen, dtype=jnp.float32)

    # angles[m, i] = m * θ_i, shape (seqlen, half)
    angles = jnp.outer(positions, freqs)
    cos = jnp.cos(angles)  # (seqlen, half)
    sin = jnp.sin(angles)  # (seqlen, half)

    # Broadcast to (1, seqlen, 1, half) for batch and head dimensions
    cos = cos[None, :, None, :]
    sin = sin[None, :, None, :]

    # Split into true even/odd dimension pairs: (0,1), (2,3), ...
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]

    # 2D rotation per pair: [even, odd] -> [even*cos - odd*sin, even*sin + odd*cos]
    x_rot_even = x_even * cos - x_odd * sin
    x_rot_odd = x_even * sin + x_odd * cos

    # Interleave rotated even/odd channels back to the original head_dim layout.
    x_rot = jnp.stack([x_rot_even, x_rot_odd], axis=-1)
    return jnp.reshape(x_rot, x.shape)


# =============================================================================
# Attention Block
# =============================================================================


class GroupedQueryAttention(nnx.Module):
    """
    Minimal causal self-attention with grouped-query heads.

    Paper-aligned design choices used here:
    - Query heads and KV heads can be different (GQA)
    - Rotary Position Embeddings (RoPE) applied to Q and K
    - No dropout
    - Bias-free linear projections by default

    Args:
        d_model: Hidden/model dimension.
        num_query_heads: Number of query heads.
        num_kv_heads: Number of key/value heads.
        head_dim: Per-head channel dimension.
        use_bias: Whether projection layers use bias.
    """

    def __init__(
        self,
        rngs: nnx.Rngs,
        d_model: int,
        num_query_heads: int,
        num_kv_heads: int,
        head_dim: int,
        use_bias: bool = False,
    ):
        self.d_model = d_model
        self.num_query_heads = num_query_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = head_dim
        self.use_bias = use_bias

        assert self.num_query_heads % self.num_kv_heads == 0, (
            "num_query_heads must be divisible by num_kv_heads for GQA"
        )
        assert self.head_dim % 2 == 0, "head_dim must be even for RoPE"
        assert self.num_query_heads * self.head_dim == self.d_model, (
            "d_model must equal num_query_heads * head_dim"
        )

        # Project input tokens into Q, K, and V spaces.
        self.q_proj = nnx.Linear(
            self.d_model,
            self.num_query_heads * self.head_dim,
            use_bias=self.use_bias,
            rngs=rngs,
        )
        self.k_proj = nnx.Linear(
            self.d_model,
            self.num_kv_heads * self.head_dim,
            use_bias=self.use_bias,
            rngs=rngs,
        )
        self.v_proj = nnx.Linear(
            self.d_model,
            self.num_kv_heads * self.head_dim,
            use_bias=self.use_bias,
            rngs=rngs,
        )

        # Project attention output back to model dimension.
        self.out_proj = nnx.Linear(
            self.num_query_heads * self.head_dim,
            self.d_model,
            use_bias=self.use_bias,
            rngs=rngs,
        )

    def __call__(self, x: jax.Array) -> jax.Array:
        """
        Args:
            x: Token hidden states, shape (batch, seqlen, d_model)
        Returns:
            Output hidden states, shape (batch, seqlen, d_model)
        """
        batch, seqlen, d_model = x.shape
        assert d_model == self.d_model, "Input d_model does not match module config"

        # 1) Compute Q, K, V projections.
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # 2) Expose head dimensions.
        q = jnp.reshape(q, (batch, seqlen, self.num_query_heads, self.head_dim))
        k = jnp.reshape(k, (batch, seqlen, self.num_kv_heads, self.head_dim))
        v = jnp.reshape(v, (batch, seqlen, self.num_kv_heads, self.head_dim))

        # 3) Apply RoPE to Q and K (after head split, before GQA expansion).
        # RoPE encodes positions via rotation — applied per head, on the last dim.
        # V is NOT rotated: position encoding only affects attention score computation.
        q = _apply_rope(q)
        k = _apply_rope(k)

        # 4) Expand KV heads so each query head can attend to a matching KV head.
        kv_repeat = self.num_query_heads // self.num_kv_heads
        k = jnp.repeat(k, kv_repeat, axis=2)
        v = jnp.repeat(v, kv_repeat, axis=2)

        # 5) Move heads before sequence for standard attention math.
        q = jnp.transpose(q, (0, 2, 1, 3))  # (batch, heads, seqlen, head_dim)
        k = jnp.transpose(k, (0, 2, 1, 3))
        v = jnp.transpose(v, (0, 2, 1, 3))

        # 6) Dot-product attention scores.
        scale = 1.0 / jnp.sqrt(self.head_dim)
        scores = jnp.einsum("bhqd,bhkd->bhqk", q, k) * scale

        # 7) Causal mask: position i cannot see future position j > i.
        causal_mask = jnp.tril(jnp.ones((seqlen, seqlen), dtype=bool))
        scores = jnp.where(causal_mask[None, None, :, :], scores, -1e30)

        # 8) Softmax over keys, then weighted sum of values.
        attn = jax.nn.softmax(scores, axis=-1)
        context = jnp.einsum("bhqk,bhkd->bhqd", attn, v)

        # 9) Merge heads and project to model space.
        context = jnp.transpose(context, (0, 2, 1, 3))
        context = jnp.reshape(
            context, (batch, seqlen, self.num_query_heads * self.head_dim)
        )
        out = self.out_proj(context)
        return out


In [ ]:
"""
Minimal Mamba-2 Implementation in JAX/Flax NNX

Based on: "Transformers are SSMs: Generalized Models and Efficient Algorithms
           Through Structured State Space Duality" (Dao & Gu, 2024)

Reference code: https://github.com/state-spaces/mamba
  - mamba_ssm/modules/ssd_minimal.py  (Listing 1 from the paper)
  - mamba_ssm/modules/mamba2_simple.py (Full Mamba-2 block)

This implementation prioritizes simplicity and readability over performance.
"""

# =============================================================================
# Core SSD Algorithm (Listing 1 from the paper)
# =============================================================================


def segsum(x: jax.Array) -> jax.Array:
    """
    Stable segment sum calculation.

    Computes a lower-triangular matrix L where L[i,j] = sum(x[j:i]) for i >= j.
    This is used to build the decay matrix for the diagonal (intra-chunk) blocks
    of the structured state space model.

    In the SSM context, x contains log-space decay factors (A values), so:
      L[i,j] = sum of A[k] for k in [j, i)
    After exponentiation, exp(L[i,j]) gives the total decay from position j to i.

    Args:
        x: Decay factors, shape (..., T)
    Returns:
        Lower-triangular segment sums, shape (..., T, T)
    """
    T = x.shape[-1]

    # Step 1: Broadcast x into a T x T matrix by repeating along a new axis.
    # Each row becomes a copy of x. Shape: (..., T, T)
    x = jnp.repeat(x[..., None], T, axis=-1)

    # Step 2: Zero out the upper triangle (excluding diagonal).
    # We only want to accumulate values strictly below the diagonal.
    # mask[i,j] = True if i > j (strict lower triangle)
    mask = jnp.tril(jnp.ones((T, T), dtype=bool), k=-1)
    x = jnp.where(mask, x, 0.0)

    # Step 3: Cumulative sum along rows (axis=-2).
    # After this, position (i, j) contains sum of x[k] for k in [j, i).
    x_segsum = jnp.cumsum(x, axis=-2)

    # Step 4: Mask out the upper triangle with -inf.
    # When we later do exp(segsum), -inf -> 0, ensuring causality.
    # We keep the diagonal (k=0) since exp(0) = 1 (no self-decay).
    mask_diag = jnp.tril(jnp.ones((T, T), dtype=bool), k=0)
    x_segsum = jnp.where(mask_diag, x_segsum, -jnp.inf)

    return x_segsum


def ssd_minimal_discrete(
    X: jax.Array, A: jax.Array, B: jax.Array, C: jax.Array, block_len: int
) -> jax.Array:
    """
    Structured State Space Duality (SSD) algorithm — Listing 1 from the paper.

    This implements the SSM recurrence h[t] = A[t]*h[t-1] + B[t]*x[t] efficiently
    by chunking the sequence and factorizing the computation into:
      1. Intra-chunk outputs  (diagonal blocks  — within each chunk)
      2. Intra-chunk states   (right factor     — B terms)
      3. Inter-chunk states   (middle factor    — A terms, chunk-level recurrence)
      4. State-to-output      (left factor      — C terms)

    The key insight: the SSM output matrix has a semi-separable structure that can
    be decomposed into diagonal blocks (handled by step 1) and off-diagonal blocks
    (handled by steps 2-4 via low-rank factorization).

    Args:
        X: Input values (already multiplied by dt), shape (batch, length, n_heads, headdim)
        A: Discrete decay factors (already multiplied by dt), shape (batch, length, n_heads)
        B: Input-to-state matrix (keys in attention analogy), shape (batch, length, n_heads, d_state)
        C: State-to-output matrix (queries in attention analogy), shape (batch, length, n_heads, d_state)
        block_len: Chunk size Q for splitting the sequence

    Returns:
        Y: Output, shape (batch, length, n_heads, headdim)
    """
    batch, length, n_heads, headdim = X.shape
    d_state = B.shape[-1]

    assert length % block_len == 0, (
        f"Length {length} must be divisible by block_len {block_len}"
    )
    n_chunks = length // block_len

    # Reshape the sequence into chunks: (batch, n_chunks, block_len, ...)
    X = jnp.reshape(X, (batch, n_chunks, block_len, n_heads, headdim))
    A = jnp.reshape(A, (batch, n_chunks, block_len, n_heads))
    B = jnp.reshape(B, (batch, n_chunks, block_len, n_heads, d_state))
    C = jnp.reshape(C, (batch, n_chunks, block_len, n_heads, d_state))

    # Rearrange A to (batch, n_heads, n_chunks, block_len) for segment sums
    A = jnp.transpose(A, (0, 3, 1, 2))

    # Cumulative sum of A within each chunk — used to compute decays
    A_cumsum = jnp.cumsum(A, axis=-1)

    # ---- Step 1: Intra-chunk outputs (diagonal blocks) ----
    # Build the T×T causal decay matrix within each chunk using segsum.
    # L[i,j] = exp(sum of A[k] for k in [j,i)), i.e. the decay from j to i.
    # Then compute: Y_diag = C @ diag(L) @ B^T @ X  (within each chunk)
    L = jnp.exp(segsum(A))
    Y_diag = jnp.einsum("bclhn,bcshn,bhcls,bcshp->bclhp", C, B, L, X)

    # ---- Step 2: Intra-chunk states (right factor of off-diagonal blocks) ----
    # Compute the SSM state at the END of each chunk by accumulating inputs B*X
    # weighted by their decay to the chunk boundary.
    # decay_states[t] = exp(A_cumsum[-1] - A_cumsum[t]) = decay from t to end of chunk
    decay_states = jnp.exp((A_cumsum[:, :, :, -1:] - A_cumsum))
    states = jnp.einsum("bclhn,bhcl,bclhp->bchpn", B, decay_states, X)

    # ---- Step 3: Inter-chunk SSM recurrence (middle factor) ----
    # Propagate states across chunk boundaries using chunk-level decays.
    # Each chunk's output state decays into the next chunk's input state.
    # This is the "recurrence across chunks" part.
    initial_states = jnp.zeros_like(states[:, :1])
    states = jnp.concatenate([initial_states, states], axis=1)

    # Build chunk-level decay matrix using segsum on the total decay per chunk
    # A_cumsum[..., -1] = total decay within each chunk
    # Pad with a leading 0 to account for the initial state
    A_cumsum_last = A_cumsum[..., -1]
    A_cumsum_last_padded = jnp.pad(A_cumsum_last, ((0, 0), (0, 0), (1, 0)))
    decay_chunk = jnp.exp(segsum(A_cumsum_last_padded))

    # Apply chunk-level recurrence: propagate states through all previous chunks
    new_states = jnp.einsum("bhzc,bchpn->bzhpn", decay_chunk, states)
    states = new_states[:, :-1]  # Drop the last (it would be the "next" state)

    # ---- Step 4: State-to-output per chunk (left factor of off-diagonal blocks) ----
    # Convert each chunk's incoming state to per-element outputs.
    # state_decay_out[t] = exp(A_cumsum[t]) = decay from chunk start to position t
    state_decay_out = jnp.exp(A_cumsum)
    Y_off = jnp.einsum("bclhn,bchpn,bhcl->bclhp", C, states, state_decay_out)

    # ---- Combine intra-chunk and inter-chunk outputs ----
    Y = Y_diag + Y_off
    Y = jnp.reshape(Y, (batch, length, n_heads, headdim))
    return Y


# =============================================================================
# Mamba-2 Block (based on mamba2_simple.py)
# =============================================================================


class Mamba2Block(nnx.Module):
    """
    A single Mamba-2 block.

    Architecture:
        input -> in_proj -> [z (gate), xBC, dt]
                              |
                        causal conv1d + SiLU
                              |
                         split [x, B, C]
                              |
                        SSD(x*dt, A*dt, B, C) + D*x
                              |
                         gate (z) + norm
                              |
                        out_proj -> output

    This mirrors the official Mamba2Simple module but uses Flax NNX instead
    of PyTorch, and calls ssd_minimal_discrete instead of CUDA/Triton kernels.
    """

    def __init__(
        self,
        rngs: nnx.Rngs,
        d_model: int,  # Model dimension
        d_state: int = 64,  # SSM state dimension (N in the paper)
        d_conv: int = 4,  # Causal convolution kernel width
        expand: int = 2,  # Expansion factor for inner dimension
        headdim: int = 64,  # Dimension per head (P in the paper)
        ngroups: int = 1,  # Number of groups for B,C (like grouped-query attention)
        chunk_size: int = 64,  # Chunk size for SSD algorithm
    ):
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = expand
        self.d_inner = self.expand * self.d_model  # D in the paper
        self.headdim = headdim
        self.ngroups = ngroups
        self.chunk_size = chunk_size

        assert self.d_inner % self.headdim == 0, "d_inner must be divisible by headdim"
        self.nheads = self.d_inner // self.headdim  # H in the paper
        assert self.nheads % self.ngroups == 0, "nheads must be divisible by ngroups"

        # --- Learnable Parameters ---

        # 1. Input projection: projects d_model -> [z, x, B, C, dt] all at once
        #    z:  gating branch,         size = d_inner
        #    xBC: conv input branch,    size = d_inner + 2*ngroups*d_state
        #    dt: step size per head,    size = nheads
        d_in_proj = 2 * self.d_inner + 2 * self.ngroups * self.d_state + self.nheads
        self.in_proj = nnx.Linear(self.d_model, d_in_proj, use_bias=False, rngs=rngs)

        # 2. Depthwise causal 1D convolution on [x, B, C]
        #    Each channel is convolved independently (feature_group_count = conv_dim)
        conv_dim = self.d_inner + 2 * self.ngroups * self.d_state
        self.conv1d = nnx.Conv(
            in_features=conv_dim,
            out_features=conv_dim,
            kernel_size=(d_conv,),
            feature_group_count=conv_dim,  # depthwise: each channel has its own filter
            use_bias=True,
            padding="VALID",  # We handle causal padding manually
            rngs=rngs,
        )

        # 3. dt (step size) bias — added before softplus activation
        self.dt_bias = nnx.Param(jnp.zeros((self.nheads,)))

        # 4. A parameter — log-parameterized so exp(A_log) > 0, then negated for decay
        #    Initialized as log(1), log(2), ..., log(nheads) following the paper
        A = jnp.arange(1, self.nheads + 1, dtype=jnp.float32)
        self.A_log = nnx.Param(jnp.log(A))

        # 5. D parameter — skip connection (like a residual from input to output)
        self.D = nnx.Param(jnp.ones((self.nheads,)))

        # 6. Output normalization + projection.
        #    The official Mamba-2 code uses RMSNormGated with norm_before_gate=False,
        #    meaning: apply the gate (y * silu(z)) first, then normalize.
        #    nnx.RMSNorm matches this: it has no bias term (unlike LayerNorm),
        #    which is consistent with the paper's bias-free design.
        self.norm = nnx.RMSNorm(self.d_inner, rngs=rngs)
        self.out_proj = nnx.Linear(
            self.d_inner, self.d_model, use_bias=False, rngs=rngs
        )

    def __call__(self, u: jax.Array) -> jax.Array:
        """
        Forward pass.

        Args:
            u: Input tensor of shape (batch, seqlen, d_model)
        Returns:
            Output tensor of shape (batch, seqlen, d_model)
        """
        batch, seqlen, _ = u.shape

        # --- 1. Input Projection ---
        # Project input to get all branches in one matrix multiply
        zxbcdt = self.in_proj(u)  # (batch, seqlen, d_in_proj)

        # Split into: z (gate), xBC (conv input), dt (step sizes)
        # Note: jnp.split takes split *indices*, not sizes (unlike torch.split)
        z, xBC, dt = jnp.split(
            zxbcdt,
            [self.d_inner, 2 * self.d_inner + 2 * self.ngroups * self.d_state],
            axis=-1,
        )
        # z:   (batch, seqlen, d_inner)     — gating signal
        # xBC: (batch, seqlen, conv_dim)    — will become x, B, C after conv
        # dt:  (batch, seqlen, nheads)      — per-head step sizes

        # Apply softplus to dt (ensures positive step sizes)
        dt = jax.nn.softplus(dt + self.dt_bias.get_value())  # (batch, seqlen, nheads)

        # --- 2. Causal 1D Convolution ---
        # Pad on the left for causal masking: output at time t only sees t-d_conv+1..t
        xBC_padded = jnp.pad(xBC, ((0, 0), (self.d_conv - 1, 0), (0, 0)))
        xBC = self.conv1d(xBC_padded)
        xBC = jax.nn.silu(xBC)  # SiLU/Swish activation

        # Split conv output into x (values), B (keys), C (queries)
        x, B, C = jnp.split(
            xBC, [self.d_inner, self.d_inner + self.ngroups * self.d_state], axis=-1
        )
        # x: (batch, seqlen, d_inner)
        # B: (batch, seqlen, ngroups * d_state)
        # C: (batch, seqlen, ngroups * d_state)

        # Reshape to expose head structure
        x = jnp.reshape(x, (batch, seqlen, self.nheads, self.headdim))
        B = jnp.reshape(B, (batch, seqlen, self.ngroups, self.d_state))
        C = jnp.reshape(C, (batch, seqlen, self.ngroups, self.d_state))

        # Expand B, C from ngroups to nheads (grouped-query attention analogy)
        # If ngroups=1, each group is shared across all heads
        B = jnp.repeat(
            B, self.nheads // self.ngroups, axis=2
        )  # (batch, seqlen, nheads, d_state)
        C = jnp.repeat(
            C, self.nheads // self.ngroups, axis=2
        )  # (batch, seqlen, nheads, d_state)

        # --- 3. SSD Core Computation ---
        # Continuous-time A: always negative (ensures decay/stability)
        A = -jnp.exp(self.A_log.get_value())  # (nheads,)

        # Discretize: multiply by step size dt
        # This converts continuous A to discrete A_bar = exp(A * dt)
        # (in log space, we just multiply and later exponentiate inside SSD)
        A_discrete = A * dt  # (batch, seqlen, nheads)
        X = x * dt[..., None]  # (batch, seqlen, nheads, headdim) — discretized input

        # Run the chunked SSD algorithm
        y = ssd_minimal_discrete(X, A_discrete, B, C, self.chunk_size)

        # Add D skip connection: D * x (direct input-to-output path)
        y = y + self.D.get_value()[None, None, :, None] * x

        # Flatten heads back to d_inner
        y = jnp.reshape(y, (batch, seqlen, self.d_inner))

        # --- 4. Gating, Normalization, and Output Projection ---
        # Gate: element-wise multiply with SiLU-activated z branch
        # (norm_before_gate=False in the official code: gate first, then norm)
        y = y * jax.nn.silu(z)
        y = self.norm(y)

        out = self.out_proj(y)
        return out


In [ ]:
"""
Minimal Sparse Mixture-of-Experts (MoE) in JAX/Flax NNX.

Based on Nemotron 3 Nano (arXiv:2512.20848, §2.1).

Key design choices from the paper:

1. Granular routed experts (DeepSeekMoE style, Dai et al. 2024):
   Each "base" expert is split into finer-grained smaller experts.
   Total routed experts = num_experts * granularity_factor,
   each with hidden_dim = expert_hidden_dim / granularity_factor.
   This improves expert specialization without changing total parameter count.

2. Shared experts:
   Always-on FFN experts that run for every token, unconditionally.
   They provide a stable shared capacity outside of routing competition.
   Shared experts keep the full expert_hidden_dim.

3. Sigmoid gating (Nemotron-specific, unlike most MoE models that use softmax):
   Gate scores are produced independently per expert via sigmoid.
   This means experts do NOT compete with each other for probability mass.
   After top-k selection, selected scores are renormalized to sum to 1
   so that the combined output has a stable scale.

4. Squared-ReLU activation inside each expert FFN:
   relu(x)^2 — a stronger nonlinearity than plain ReLU.

5. Aux-loss-free load balancing (Wang et al. 2024, as used in Nemotron 3 Nano §2.4):
   Instead of an auxiliary gradient loss, each expert gets a learnable bias term.
   The bias is added to routing scores at selection time, nudging the router toward
   under-utilized experts — WITHOUT affecting the actual output gate weights.
   After each training step, biases are updated with a simple sign rule:
     if expert i got too many tokens → decrease its bias (harder to pick next time)
     if expert i got too few tokens  → increase its bias (easier to pick next time)
   The update rate is 1e-3 per the paper. This produces no extra gradient computation.

6. No bias on any linear layers (per paper).

The implementation is explicit and loop-based for readability, not performance.
"""

# =============================================================================
# Sparse MoE Block
# =============================================================================


class SparseMoE(nnx.Module):
    """
    Sparse MoE layer matching the Nemotron 3 Nano design (arXiv:2512.20848, §2.1).

    --- Granular experts (DeepSeekMoE style) ---
    Instead of a few large experts, we use many small, fine-grained experts.
    Given `num_experts` base experts and a `granularity_factor` g:
      - Actual routed expert count = num_experts * g
      - Each routed expert hidden dim = expert_hidden_dim / g
      - Top-k = top_k * g   (if scale_top_k_with_granularity=True)
    Total FLOPs per token stays the same, but with more diverse expert paths.
    In Nemotron 3 Nano: 128 total routable experts, 6 activated per token.

    --- Shared experts ---
    `num_shared_experts` always-on FFN experts run on every token.
    They are NOT subject to routing — they always contribute to the output.
    Their outputs are summed and added to the routed path output.
    Shared experts keep the full expert_hidden_dim (not reduced by granularity).
    In Nemotron 3 Nano: 2 shared experts.

    --- Sigmoid routing (Nemotron-specific) ---
    Router logits -> sigmoid -> top-k selection.
    With softmax (most MoEs): experts compete; picking one raises another's cost.
    With sigmoid: scores are independent; each expert is scored on its own merit.
    After top-k, selected scores are renormalized to sum to 1 for stable output scale.

    --- Aux-loss-free load balancing ---
    Without any balancing, the router collapses: it always picks a few favourite
    experts and ignores the rest. The fix used in Nemotron 3 Nano is NOT a loss
    term — instead, each expert gets a persistent bias scalar.

    At routing time: top-k uses (sigmoid_score + expert_bias) to decide which
    experts to activate. A higher bias makes an expert easier to pick.

    At output time: gate weights use the ORIGINAL sigmoid scores (without bias),
    so the learned expert magnitudes are preserved.

    After every training step (outside the gradient): biases are nudged with a
    simple sign update. Overloaded experts get a smaller bias; underloaded ones
    get a larger bias. Over time this pushes the router toward balance.

    Args:
        granularity_factor:
            1 = standard MoE (no granularity).
            >1 = each base expert is split into this many smaller experts.
        scale_top_k_with_granularity:
            True (default): effective top-k = top_k * granularity_factor.
            False: keep top-k fixed regardless of granularity.
        bias_update_rate:
            Step size for the expert bias update. 1e-3 per Nemotron 3 Nano §2.4.
    """

    def __init__(
        self,
        rngs: nnx.Rngs,
        d_model: int,
        num_experts: int,
        num_shared_experts: int,
        top_k: int,
        expert_hidden_dim: int,
        use_bias: bool = False,
        granularity_factor: int = 1,
        scale_top_k_with_granularity: bool = True,
        bias_update_rate: float = 1e-3,  # Wang et al. 2024 / Nemotron 3 Nano §2.4
    ):
        self.d_model = d_model

        # Keep base values for reference.
        self.num_experts = num_experts
        self.top_k = top_k
        self.num_shared_experts = num_shared_experts
        self.expert_hidden_dim = expert_hidden_dim

        self.granularity_factor = granularity_factor
        self.scale_top_k_with_granularity = scale_top_k_with_granularity

        assert self.num_experts > 0, "num_experts must be > 0"
        assert self.top_k > 0, "top_k must be > 0"
        assert self.top_k <= self.num_experts, "top_k must be <= num_experts"
        assert self.num_shared_experts >= 0, "num_shared_experts must be >= 0"
        assert self.granularity_factor > 0, "granularity_factor must be > 0"

        # Total fine-grained routed experts after granular splitting.
        # e.g. 16 base experts * 8 granularity_factor = 128 (as in Nemotron 3 Nano).
        self.num_routed_experts = self.num_experts * self.granularity_factor

        # Scale top-k proportionally so the same fraction of capacity is activated.
        # e.g. top_k=1 with granularity=6 → select 6 out of 128 experts.
        if self.scale_top_k_with_granularity:
            self.routed_top_k = self.top_k * self.granularity_factor
        else:
            self.routed_top_k = self.top_k

        assert self.routed_top_k <= self.num_routed_experts, (
            "effective routed top-k must be <= num_routed_experts"
        )

        # Granular routed experts are narrower to keep total parameter count stable.
        # e.g. expert_hidden_dim=1856, granularity=8 → each expert hidden = 232.
        self.routed_expert_hidden_dim = max(
            1, self.expert_hidden_dim // self.granularity_factor
        )

        # Shared experts keep the full hidden dimension.
        # They're meant to model general token features, so they stay large.
        self.shared_expert_hidden_dim = self.expert_hidden_dim

        # Step size used when updating expert biases after each training step.
        self.bias_update_rate = bias_update_rate

        # Expert bias for aux-loss-free load balancing.
        # Shape: (num_routed_experts,), initialized to 0 for all experts equally.
        # NOT updated by the gradient optimizer — updated manually via update_expert_bias().
        # Stored as a plain nnx.Variable (not nnx.Param) so it can be filtered out
        # of gradient updates when extracting model parameters.
        self.expert_bias = nnx.Variable(jnp.zeros(self.num_routed_experts))

        # Stores the top-k expert indices from the most recent forward pass.
        # Shape at runtime: (num_tokens, routed_top_k). Placeholder shape here.
        # After each training step, the training loop reads this to call
        # update_expert_bias(self.last_topk_indices.value) — outside the gradient.
        self.last_topk_indices = nnx.Variable(
            jnp.zeros((1, self.routed_top_k), dtype=jnp.int32)
        )

        # Router: a single linear layer mapping each token to one logit per routed expert.
        # No bias per paper. Shared experts are NOT routed — they bypass this.
        self.router = nnx.Linear(
            self.d_model,
            self.num_routed_experts,
            use_bias=use_bias,
            rngs=rngs,
        )

        # All routed expert weights are stored pre-stacked so _collect_routed_outputs
        # can index directly — no jnp.stack() assembly on every forward pass.
        # routed_W1: (num_routed_experts, d_model, routed_expert_hidden_dim)
        # routed_W2: (num_routed_experts, routed_expert_hidden_dim, d_model)
        # No bias on any linear layers, per paper.
        init = nnx.initializers.lecun_normal()
        self.routed_W1 = nnx.Param(
            init(
                rngs.params(),
                (self.num_routed_experts, d_model, self.routed_expert_hidden_dim),
            )
        )
        self.routed_W2 = nnx.Param(
            init(
                rngs.params(),
                (self.num_routed_experts, self.routed_expert_hidden_dim, d_model),
            )
        )

        # Shared expert weights are also pre-stacked for the batched einsum in
        # _collect_shared_outputs. Shared experts keep the full hidden dimension.
        # shared_W1: (num_shared_experts, d_model, shared_expert_hidden_dim)
        # shared_W2: (num_shared_experts, shared_expert_hidden_dim, d_model)
        if self.num_shared_experts > 0:
            self.shared_W1 = nnx.Param(
                init(
                    rngs.params(),
                    (self.num_shared_experts, d_model, self.shared_expert_hidden_dim),
                )
            )
            self.shared_W2 = nnx.Param(
                init(
                    rngs.params(),
                    (self.num_shared_experts, self.shared_expert_hidden_dim, d_model),
                )
            )

    def _collect_routed_outputs(
        self,
        x_flat: jax.Array,
        topk_indices: jax.Array,
    ) -> jax.Array:
        """
        Run only the selected routed experts via pre-stacked weight gather + einsum.

        Weights are stored pre-stacked as (num_routed_experts, ...) tensors in
        __init__, so we index directly with topk_indices — no per-forward assembly.
        Only (num_tokens × routed_top_k) expert forward passes are computed;
        non-selected experts are never touched.

        Args:
            x_flat: (num_tokens, d_model)
            topk_indices: (num_tokens, routed_top_k)
        Returns:
            routed_outputs: (num_tokens, routed_top_k, d_model)
                            only the selected expert outputs, in top-k order.
        """
        # Weights are pre-stacked at init time — no assembly cost here.
        # W1: (num_routed_experts, d_model,      routed_expert_hidden_dim)
        # W2: (num_routed_experts, routed_expert_hidden_dim, d_model)
        W1 = self.routed_W1.get_value()
        W2 = self.routed_W2.get_value()

        # Gather only the weights of each token's top-k selected experts.
        # topk_indices: (num_tokens, routed_top_k)  →  index into axis-0 of W1/W2
        # W1_sel: (num_tokens, routed_top_k, d_model,      hidden_dim)
        # W2_sel: (num_tokens, routed_top_k, hidden_dim,   d_model)
        W1_sel = W1[topk_indices]
        W2_sel = W2[topk_indices]

        # fc1: apply each token's K selected up-projections simultaneously.
        # 't d, t k d h -> t k h'  means: for every token t and selected expert k,
        # dot x_flat[t] with W1_sel[t, k] to get a hidden vector of size h.
        h = jnp.einsum("td,tkdh->tkh", x_flat, W1_sel)

        # Squared-ReLU activation: relu(x)^2  (paper §2.1 / MoEExpert.__call__).
        h = jax.nn.relu(h)
        h = h * h

        # fc2: apply each token's K selected down-projections.
        # 't k h, t k h d -> t k d'  projects hidden vectors back to d_model.
        out = jnp.einsum("tkh,tkhd->tkd", h, W2_sel)
        # out: (num_tokens, routed_top_k, d_model)
        return out

    def _collect_shared_outputs(self, x_flat: jax.Array) -> jax.Array:
        """
        Run all shared experts on every token via a single batched einsum.

        Shared experts are always active — no routing decision is made.
        All experts are applied in parallel using pre-stacked weight matrices,
        producing one output per expert per token in a single fused operation.

        Args:
            x_flat: (num_tokens, d_model)
        Returns:
            shared_outputs: (num_tokens, num_shared_experts, d_model),
                            or (num_tokens, 0, d_model) if there are no shared experts.
        """
        if self.num_shared_experts == 0:
            return jnp.zeros((x_flat.shape[0], 0, self.d_model), dtype=x_flat.dtype)

        # All shared experts run in one batched einsum — no sequential loop.
        # 'td,edh->teh': for each of the E shared experts, project every token
        # from d_model up to shared_expert_hidden_dim simultaneously.
        h = jnp.einsum("td,edh->teh", x_flat, self.shared_W1.get_value())
        # Squared-ReLU: relu(x)^2 — same activation as the routed experts.
        h = jax.nn.relu(h)
        h = h * h
        # 'teh,ehd->ted': project all shared expert hidden states back to d_model.
        # out: (num_tokens, num_shared_experts, d_model)
        return jnp.einsum("teh,ehd->ted", h, self.shared_W2.get_value())

    def update_expert_bias(self, topk_indices: jax.Array) -> None:
        """
        Update the expert bias after each training step.

        This is the core of the aux-loss-free load balancing strategy
        (Wang et al. 2024), as used in Nemotron 3 Nano (§2.4).

        The idea is simple:
          - Count how many tokens each expert received in this step.
          - Compare to the ideal uniform count (tokens * top_k / num_experts).
          - Decrease the bias of overloaded experts so they win fewer top-k races.
          - Increase the bias of underloaded experts so they win more top-k races.

        The update uses sign() instead of the actual count difference, so the
        step size is always exactly +/- bias_update_rate regardless of how far
        off-balance the expert is. This keeps the bias values small and stable.

        IMPORTANT: Call this AFTER the optimizer step, outside the gradient tape.
        The expert_bias is NOT a gradient parameter — it must not be passed to
        the optimizer. Filter it out by type when building the optimizer state.

        Args:
            topk_indices: (num_tokens, routed_top_k) from the last forward pass.
                          These are the expert indices that were selected.
        """
        num_tokens = topk_indices.shape[0]

        # Count how many tokens were routed to each expert.
        # one_hot: (num_tokens, routed_top_k, num_routed_experts)
        # sum over (tokens, top_k slots) → (num_routed_experts,)
        actual_count = jnp.sum(
            jax.nn.one_hot(topk_indices, self.num_routed_experts),
            axis=(0, 1),
        )

        # The ideal count if all experts were equally loaded.
        expected_count = num_tokens * self.routed_top_k / self.num_routed_experts

        # sign(actual - expected):
        #   +1 means overloaded  → subtract from bias → harder to pick next time
        #   -1 means underloaded → add to bias        → easier to pick next time
        #    0 means perfect     → no change
        self.expert_bias.set_value(
            self.expert_bias.get_value()
            - self.bias_update_rate * jnp.sign(actual_count - expected_count)
        )

    def __call__(self, x: jax.Array) -> jax.Array:
        """
        Forward pass through the sparse MoE layer.

        Routing overview (aux-loss-free approach):
            1. Router produces one sigmoid score per expert, per token.
            2. Expert bias is ADDED to scores for top-k selection only.
               This steers the router toward under-utilized experts.
            3. Top-k selection uses the biased scores to choose which experts run.
            4. Gate weights use the ORIGINAL (unbiased) sigmoid scores, renormalized.
               The bias is a routing hint, not a magnitude signal.
            5. Weighted sum of selected expert outputs forms the routed path.
            6. All shared experts run unconditionally; their outputs are summed in.

        To balance load: call update_expert_bias(topk_indices) after each training step.

        Args:
            x: (batch, seqlen, d_model)

        Returns:
            y: (batch, seqlen, d_model)
            jnp.zeros(()) (optional): only returned when return_aux_loss=True
        """
        batch, seqlen, d_model = x.shape
        assert d_model == self.d_model, "Input d_model does not match MoE config"

        # MoE routing is purely token-wise, so we flatten batch and sequence together.
        num_tokens = batch * seqlen
        x_flat = jnp.reshape(x, (num_tokens, d_model))  # (num_tokens, d_model)

        # ── Routed path ────────────────────────────────────────────────────────

        # Step 1: Compute one routing logit per expert for each token.
        routed_logits = self.router(x_flat)  # (num_tokens, num_routed_experts)

        # Step 2: Apply sigmoid to get independent gate scores.
        # Unlike softmax, sigmoid does NOT create a probability distribution.
        # Each expert's score is judged independently — scores do not compete.
        routed_scores = jax.nn.sigmoid(
            routed_logits
        )  # (num_tokens, num_routed_experts)

        # Step 3 (aux-loss-free): Add the expert bias to scores before top-k selection.
        # The bias is learned over time: underloaded experts accumulate a positive bias
        # (making them easier to pick), overloaded experts accumulate a negative bias
        # (making them harder to pick). This is the key load-balancing mechanism.
        # expert_bias shape: (num_routed_experts,) → broadcasts across all tokens.
        biased_scores = routed_scores + self.expert_bias.get_value()

        # Step 4: Select top-k experts using BIASED scores.
        # We use biased scores here so the selection reflects the desired load balance.
        # topk_indices: (num_tokens, routed_top_k) — which expert indices were chosen
        _, topk_indices = jax.lax.top_k(biased_scores, self.routed_top_k)

        # Save topk_indices so the training loop can call update_expert_bias()
        # AFTER the optimizer step, outside the gradient computation.
        self.last_topk_indices.set_value(topk_indices)

        # Step 5: Build gate weights using the ORIGINAL (unbiased) sigmoid scores.
        # The bias only determines WHO gets selected, not HOW MUCH they contribute.
        # Using original scores preserves the expert's learned signal magnitude.
        # Gather only the top-k scores — no need to build a full (num_tokens, num_routed_experts) tensor.
        token_ids = jnp.arange(num_tokens)[:, None]  # (num_tokens, 1)
        selected_scores = routed_scores[
            token_ids, topk_indices
        ]  # (num_tokens, routed_top_k)

        # Step 6: Renormalize so selected gate weights sum to 1 per token.
        # This keeps the output scale stable regardless of the absolute score values.
        selected_gates = selected_scores / (
            jnp.sum(selected_scores, axis=-1, keepdims=True) + 1e-6
        )  # (num_tokens, routed_top_k)

        # Step 7: Run only selected routed experts and compute the gated weighted sum.
        # routed_outputs: (num_tokens, routed_top_k, d_model) — already only top-k, no zeros padding.
        routed_outputs = self._collect_routed_outputs(x_flat, topk_indices)

        # selected_gates[:, :, None] broadcasts to (num_tokens, routed_top_k, d_model).
        routed_mix = jnp.sum(routed_outputs * selected_gates[:, :, None], axis=1)
        # routed_mix: (num_tokens, d_model)
        # routed_mix: (num_tokens, d_model)

        # ── Shared path ─────────────────────────────────────────────────────────

        if self.num_shared_experts > 0:
            # Shared experts run on every token with no gating or selection.
            shared_outputs = self._collect_shared_outputs(x_flat)
            # shared_outputs: (num_tokens, num_shared_experts, d_model)

            # Sum across shared experts: each expert adds its own contribution.
            shared_mix = jnp.sum(shared_outputs, axis=1)  # (num_tokens, d_model)

            # Final output = routed path + shared path.
            y_flat = routed_mix + shared_mix
        else:
            y_flat = routed_mix

        # Restore the original (batch, seqlen, d_model) shape.
        y = jnp.reshape(y_flat, (batch, seqlen, d_model))

        return y


In [ ]:
"""
Minimal Nemotron-style Hybrid Model in JAX/Flax NNX.

Reference inspiration:
"Nemotron 3 Nano: Open, Efficient Mixture-of-Experts Hybrid
 Mamba-Transformer Model for Agentic Reasoning"

What this minimal implementation keeps from the paper:
- Hybrid stack: Mamba mixer + Attention mixer (alternating pattern by default)
- MoE after each mixer block
- Sparse top-k MoE routing with shared experts
- Squared-ReLU experts
- RMSNorm + residual pre-norm structure
- RoPE in attention, no dropout, and bias-free linear layers by default

What is intentionally simplified:
- Tiny default dimensions for local experimentation
- Alternating layer pattern instead of large paper-scale block scheduling
- No distributed/expert-parallel optimization
"""

# =============================================================================
# Config
# =============================================================================


def _default_patterns() -> list[tuple[str, int]]:
    return [
        ("mamba_moe", 2),
        ("mamba_attention_moe", 1),
        ("mamba_moe", 2),
        ("mamba_attention_moe", 1),
        ("mamba_moe", 2),
        ("mamba_attention_moe", 1),
        ("mamba_moe", 1),
    ]


@dataclass
class NemotronConfig:
    """
    Config with tiny local defaults.

    Notes:
    - Defaults are intentionally small for easy local runs.
    - Paper-like behavior is preserved as configurable knobs.
    """

    # Token/model sizes
    vocab_size: int = 1000
    d_model: int = 128

    patterns: list[tuple[str, int]] = field(default_factory=_default_patterns)

    # Attention (GQA)
    num_attention_heads: int = 4
    num_kv_heads: int = 1
    attention_head_dim: int = 32

    # Mamba-2 settings
    mamba_d_state: int = 64
    mamba_d_conv: int = 4
    mamba_expand: int = 2
    mamba_headdim: int = 64
    mamba_ngroups: int = 1
    mamba_chunk_size: int = 64

    # MoE settings
    num_experts: int = 4
    num_shared_experts: int = 1
    top_k: int = 2
    expert_hidden_dim: int = 256
    granularity_factor: int = 1
    scale_top_k_with_granularity: bool = True

    # Normalization and numerical stability
    rms_norm_eps: float = 1e-6

    @classmethod
    def from_preset(cls, preset: str = "tiny") -> "NemotronConfig":
        """
        Builds a config from a named preset.

        Presets:
        - tiny: default local-friendly profile (fallback)
        - paper_close: larger profile that is closer to Nemotron-3-Nano style

        The paper_close preset increases attention heads, expert count, and
        uses top_k=6 routing with stronger granular MoE settings while still
        keeping this implementation simple.
        """
        key = preset.strip().lower()

        if key in ("tiny", "default"):
            return cls(
                patterns=_default_patterns(),
            )

        if key in ("paper_close", "paper-close", "paper"):
            return cls(
                # Bigger model than tiny defaults.
                d_model=2048,
                patterns=[
                    ("mamba_moe", 2),
                    ("mamba_attention_moe", 1),
                    ("mamba_moe", 2),
                    ("mamba_attention_moe", 1),
                    ("mamba_moe", 2),
                    ("mamba_attention_moe", 1),
                    ("mamba_moe", 2),
                    ("mamba_attention_moe", 1),
                    ("mamba_moe", 2),
                    ("mamba_attention_moe", 1),
                    ("mamba_moe", 3),
                    ("mamba_attention_moe", 1),
                    ("mamba_moe", 4),
                ],
                # Closer to paper-style GQA shape choices.
                num_attention_heads=32,
                num_kv_heads=2,
                attention_head_dim=64,
                # Closer to paper-style Mamba settings.
                mamba_d_state=128,
                mamba_d_conv=4,
                mamba_expand=2,
                mamba_headdim=64,
                mamba_ngroups=8,
                mamba_chunk_size=64,
                # Closer to paper-style MoE settings.
                num_experts=64,
                num_shared_experts=2,
                top_k=6,
                expert_hidden_dim=1856,
                granularity_factor=2,
                # Keep exactly 6 activated routed experts (paper behavior).
                scale_top_k_with_granularity=False,
                rms_norm_eps=1e-6,
            )

        raise ValueError(
            f"Unknown preset '{preset}'. Supported presets: tiny, paper_close"
        )

    def validate(self) -> None:
        """Checks shape constraints that must hold for this architecture."""
        assert len(self.patterns) > 0, "patterns cannot be empty"

        # Attention output must map cleanly back to d_model.
        assert self.num_attention_heads * self.attention_head_dim == self.d_model, (
            "d_model must equal num_attention_heads * attention_head_dim"
        )
        assert self.num_attention_heads % self.num_kv_heads == 0, (
            "num_attention_heads must be divisible by num_kv_heads"
        )

        # Mamba internal shape constraints.
        mamba_d_inner = self.mamba_expand * self.d_model
        assert mamba_d_inner % self.mamba_headdim == 0, (
            "(mamba_expand * d_model) must be divisible by mamba_headdim"
        )
        mamba_nheads = mamba_d_inner // self.mamba_headdim
        assert mamba_nheads % self.mamba_ngroups == 0, (
            "Mamba nheads must be divisible by mamba_ngroups"
        )

        # MoE routing constraints.
        assert self.top_k > 0, "top_k must be > 0"
        assert self.top_k <= self.num_experts, "top_k must be <= num_experts"
        assert self.granularity_factor > 0, "granularity_factor must be > 0"

        effective_num_routed_experts = self.num_experts * self.granularity_factor
        if self.scale_top_k_with_granularity:
            effective_top_k = self.top_k * self.granularity_factor
        else:
            effective_top_k = self.top_k
        assert effective_top_k <= effective_num_routed_experts, (
            "effective routed top-k must be <= effective routed experts"
        )


# =============================================================================
# Helper
# =============================================================================


def _build_mamba(config: NemotronConfig, rngs: nnx.Rngs) -> Mamba2Block:
    return Mamba2Block(
        d_model=config.d_model,
        d_state=config.mamba_d_state,
        d_conv=config.mamba_d_conv,
        expand=config.mamba_expand,
        headdim=config.mamba_headdim,
        ngroups=config.mamba_ngroups,
        chunk_size=config.mamba_chunk_size,
        rngs=rngs,
    )


def _build_moe(config: NemotronConfig, rngs: nnx.Rngs) -> SparseMoE:
    return SparseMoE(
        d_model=config.d_model,
        num_experts=config.num_experts,
        num_shared_experts=config.num_shared_experts,
        top_k=config.top_k,
        expert_hidden_dim=config.expert_hidden_dim,
        granularity_factor=config.granularity_factor,
        scale_top_k_with_granularity=config.scale_top_k_with_granularity,
        use_bias=False,
        rngs=rngs,
    )


# =============================================================================
# Mamba MoE Block
# =============================================================================


class MambaMoEBlock(nnx.Module):
    """
    Hybrid Mamba & MoE block.

    Block structure (pre-norm residual):
      x = x + Mamba(RMSNorm(x))
      x = x + MoE(RMSNorm(x))

    This keeps the architecture simple and easy to inspect.
    """

    def __init__(self, rngs: nnx.Rngs, config: NemotronConfig):
        self.norm_mamba = nnx.RMSNorm(config.d_model, rngs=rngs)
        self.norm_moe = nnx.RMSNorm(config.d_model, rngs=rngs)

        # Reuse the already-implemented Mamba-2 block
        self.mamba = _build_mamba(config=config, rngs=rngs)

        # MoE stage after every mixer layer.
        self.moe = _build_moe(config=config, rngs=rngs)

    def __call__(self, x: jax.Array) -> jax.Array:
        # Mamba residual path.
        x = x + self.mamba(self.norm_mamba(x))

        # MoE residual path.
        return x + self.moe(self.norm_moe(x))


# =============================================================================
# Mamba Attention MoE Block
# =============================================================================


class MambaAttentionMoEBlock(nnx.Module):
    """
    Hybrid Mamba, Attention & MoE block.

    Block structure (pre-norm residual):
      x = x + Mamba(RMSNorm(x))
      x = x + Attention(RMSNorm(x))
      x = x + MoE(RMSNorm(x))

    This keeps the architecture simple and easy to inspect.
    """

    def __init__(self, rngs: nnx.Rngs, config: NemotronConfig):
        self.norm_mamba = nnx.RMSNorm(config.d_model, rngs=rngs)
        self.norm_attention = nnx.RMSNorm(config.d_model, rngs=rngs)
        self.norm_moe = nnx.RMSNorm(config.d_model, rngs=rngs)

        # Reuse the already-implemented Mamba-2 block as the mixer.
        self.mamba = _build_mamba(config=config, rngs=rngs)

        self.attention = GroupedQueryAttention(
            d_model=config.d_model,
            num_query_heads=config.num_attention_heads,
            num_kv_heads=config.num_kv_heads,
            head_dim=config.attention_head_dim,
            use_bias=False,
            rngs=rngs,
        )

        # MoE stage after every mixer layer.
        self.moe = _build_moe(config=config, rngs=rngs)

    def __call__(self, x: jax.Array) -> jax.Array:
        # Mamba residual path.
        x = x + self.mamba(self.norm_mamba(x))

        # Attention residual path.
        x = x + self.attention(self.norm_attention(x))

        # MoE residual path.
        return x + self.moe(self.norm_moe(x))


# =============================================================================
# Nemotron 3 Nano Block
# =============================================================================


class NemotronNanoBlock(nnx.Module):
    """
    Minimal Nemotron-style language model.

    Layout:
      token embedding -> N x NemotronBlock -> RMSNorm -> LM head

        RoPE is applied inside attention blocks; there is no separate learned
        positional embedding table.
    """

    def __init__(self, rngs: nnx.Rngs, config: NemotronConfig):
        config.validate()
        self.config = config

        self.embedding = nnx.Embed(config.vocab_size, config.d_model, rngs=rngs)
        block_factories = {
            "mamba_moe": MambaMoEBlock,
            "mamba_attention_moe": MambaAttentionMoEBlock,
        }

        blocks: list[MambaMoEBlock | MambaAttentionMoEBlock] = []
        for block_type, repeats in config.patterns:
            block_factory = block_factories.get(block_type)
            if block_factory is None:
                raise ValueError(f"Unknown block type '{block_type}'")
            for _ in range(repeats):
                blocks.append(block_factory(config=config, rngs=rngs))
        self.blocks = nnx.List(blocks)

        self.final_norm = nnx.RMSNorm(config.d_model, rngs=rngs)

        # Untied output head (separate from embeddings).
        self.lm_head = nnx.Linear(
            config.d_model,
            config.vocab_size,
            use_bias=False,
            rngs=rngs,
        )

    def __call__(self, token_ids: jax.Array) -> jax.Array:
        x = self.embedding(token_ids)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        return logits


In [ ]:
"""
Simple, minimal, and explainable Nemotron app.

This script shows a full small workflow:
1) Load a real dataset (roneneldan/TinyStories)
2) Train a tiny Nemotron language model
3) Evaluate it with validation loss + perplexity
4) Chat with it in the terminal

Design goals:
- Keep code easy to read and modify.
- Keep control flow explicit.
- Prefer clarity over speed/optimization.
"""

# =============================================================================
# Tokenizer
# =============================================================================


def load_hf_tokenizer(
    tokenizer_name: str,
    cache_dir: str | None = None,
) -> "PreTrainedTokenizerBase":
    """
    Loads a tokenizer from Hugging Face and guarantees core special tokens.

    We ensure PAD/BOS/EOS IDs exist because batching and generation rely on them.
    """
    try:
        from transformers import AutoTokenizer
    except ImportError as exc:
        raise ImportError(
            "Hugging Face tokenizer support requires the 'transformers' package. "
            "Install it with: pip install transformers"
        ) from exc

    try:
        tokenizer = AutoTokenizer.from_pretrained(
            tokenizer_name,
            cache_dir=cache_dir,
            use_fast=True,
        )
    except Exception as exc:
        raise RuntimeError(
            f"Failed to load Hugging Face tokenizer '{tokenizer_name}'."
        ) from exc

    # Generation depends on EOS.
    if tokenizer.eos_token_id is None:
        if tokenizer.sep_token is not None:
            tokenizer.eos_token = tokenizer.sep_token
        else:
            tokenizer.add_special_tokens({"eos_token": "<eos>"})

    # We use BOS as a light conversation delimiter.
    if tokenizer.bos_token_id is None:
        if tokenizer.cls_token is not None:
            tokenizer.bos_token = tokenizer.cls_token
        else:
            tokenizer.add_special_tokens({"bos_token": "<bos>"})

    # Left padding is used for fixed-length context windows.
    if tokenizer.pad_token_id is None:
        if tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token
        else:
            tokenizer.add_special_tokens({"pad_token": "<pad>"})

    return tokenizer


def _get_special_token_ids(
    tokenizer: "PreTrainedTokenizerBase",
) -> tuple[int, int, int]:
    """Returns guaranteed integer IDs for PAD/BOS/EOS tokens."""
    if (
        tokenizer.pad_token_id is None
        or tokenizer.bos_token_id is None
        or tokenizer.eos_token_id is None
    ):
        raise ValueError(
            "Tokenizer must define pad_token_id, bos_token_id, eos_token_id"
        )

    return (
        int(tokenizer.pad_token_id),  # type: ignore[arg-type]
        int(tokenizer.bos_token_id),  # type: ignore[arg-type]
        int(tokenizer.eos_token_id),  # type: ignore[arg-type]
    )


def encode_text(
    tokenizer: "PreTrainedTokenizerBase",
    text: str,
    add_bos: bool = False,
    add_eos: bool = False,
) -> list[int]:
    """Encodes text and optionally prepends/appends BOS/EOS."""
    _, bos_id, eos_id = _get_special_token_ids(tokenizer)

    ids = list(tokenizer.encode(text, add_special_tokens=False))
    if add_bos:
        ids = [bos_id] + ids
    if add_eos:
        ids = ids + [eos_id]
    return ids


def _extract_story_text(example: dict[str, object]) -> str:
    """
    Reads one TinyStories row and returns its text.

    We check a few keys defensively so this stays easy to understand even if
    the upstream schema changes slightly.
    """
    for key in ("text", "story", "content"):
        value = example.get(key)
        if isinstance(value, str):
            cleaned = value.strip()
            if cleaned:
                return cleaned
    return ""


def _encode_segment(tokenizer: "PreTrainedTokenizerBase", text: str) -> list[int]:
    """Tokenizes raw text without adding any special tokens."""
    return list(tokenizer.encode(text, add_special_tokens=False))


def encode_text_with_assistant_mask(
    tokenizer: "PreTrainedTokenizerBase",
    text: str,
    user_role_tag: str,
    assistant_role_tag: str,
) -> tuple[list[int], list[float]]:
    """
    Encodes one role-tagged conversation and returns per-token loss mask.

    Mask semantics:
    - 1.0 for assistant content tokens
    - 0.0 for user content tokens and role tag tokens
    """
    token_ids: list[int] = []
    loss_mask: list[float] = []

    if not user_role_tag or not assistant_role_tag:
        raise ValueError("user_role_tag and assistant_role_tag must be non-empty")

    cursor = 0
    current_role: str | None = None

    while cursor < len(text):
        next_user = text.find(user_role_tag, cursor)
        next_assistant = text.find(assistant_role_tag, cursor)

        candidates = [idx for idx in (next_user, next_assistant) if idx != -1]
        next_tag_index = min(candidates) if candidates else -1

        if next_tag_index == -1:
            chunk = text[cursor:]
            if chunk:
                chunk_ids = _encode_segment(tokenizer, chunk)
                mask_value = 1.0 if current_role == "assistant" else 0.0
                token_ids.extend(chunk_ids)
                loss_mask.extend([mask_value] * len(chunk_ids))
            break

        if next_tag_index > cursor:
            chunk = text[cursor:next_tag_index]
            if chunk:
                chunk_ids = _encode_segment(tokenizer, chunk)
                mask_value = 1.0 if current_role == "assistant" else 0.0
                token_ids.extend(chunk_ids)
                loss_mask.extend([mask_value] * len(chunk_ids))

        if text.startswith(user_role_tag, next_tag_index):
            tag_text = user_role_tag
            current_role = "user"
        else:
            tag_text = assistant_role_tag
            current_role = "assistant"

        tag_ids = _encode_segment(tokenizer, tag_text)
        token_ids.extend(tag_ids)
        loss_mask.extend([0.0] * len(tag_ids))

        cursor = next_tag_index + len(tag_text)

    return token_ids, loss_mask


# =============================================================================
# Dataset
# =============================================================================


def load_tinystories_texts(
    max_stories: int,
    split: str = "train",
    cache_dir: str | None = None,
) -> list[str]:
    """
    Loads a bounded number of stories from roneneldan/TinyStories.

    Why a bounded subset?
    - Keeps the demo easy to run locally.
    - Keeps the data flow simple and inspectable.
    - Streams rows and stops early at `max_stories`.
    """
    if max_stories < 2:
        raise ValueError("max_stories must be at least 2")

    try:
        from datasets import load_dataset
    except ImportError as exc:
        raise ImportError(
            "TinyStories training requires the 'datasets' package. "
            "Install it with: pip install datasets"
        ) from exc

    try:
        dataset = load_dataset(
            "roneneldan/TinyStories",
            split=split,
            cache_dir=cache_dir,
            streaming=True,
        )
    except Exception as exc:
        raise RuntimeError(
            "Failed to load roneneldan/TinyStories. "
            "Check your internet connection and cache permissions."
        ) from exc

    stories: list[str] = []

    # Intentionally iterate in plain Python for readability.
    for example in dataset:
        story = _extract_story_text(example)
        if story:
            stories.append(story)
        if len(stories) >= max_stories:
            break

    if len(stories) < 2:
        raise ValueError("TinyStories did not return enough non-empty stories")

    return stories


def split_train_val_texts(
    stories: list[str],
    train_ratio: float,
) -> tuple[list[str], list[str]]:
    """
    Splits stories into train and validation lists.

    This uses a deterministic ordered split to keep behavior simple and
    reproducible.
    """
    if not 0.0 < train_ratio < 1.0:
        raise ValueError("train_ratio must be between 0 and 1")
    if len(stories) < 2:
        raise ValueError("Need at least 2 stories for train/validation split")

    split_index = int(len(stories) * train_ratio)
    split_index = max(1, split_index)
    split_index = min(split_index, len(stories) - 1)

    train_texts = stories[:split_index]
    val_texts = stories[split_index:]
    return train_texts, val_texts


def load_jsonl_texts(
    jsonl_path: str,
    max_records: int,
    text_key: str = "serialized_text",
) -> list[str]:
    """
    Loads text samples from a JSONL file.

    Each non-empty line must contain a JSON object with a string field at
    `text_key` (default: serialized conversation text).
    """
    if max_records <= 0:
        raise ValueError("max_records must be > 0")

    path = Path(jsonl_path)
    if not path.exists():
        raise FileNotFoundError(f"JSONL file not found: {jsonl_path}")

    texts: list[str] = []
    with path.open("r", encoding="utf-8") as infile:
        for line_number, line in enumerate(infile, start=1):
            if len(texts) >= max_records:
                break

            stripped = line.strip()
            if not stripped:
                continue

            try:
                obj = json.loads(stripped)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {jsonl_path} at line {line_number}"
                ) from exc

            value = obj.get(text_key) if isinstance(obj, dict) else None
            if isinstance(value, str):
                cleaned = value.strip()
                if cleaned:
                    texts.append(cleaned)

    if len(texts) < 2:
        raise ValueError(
            f"Need at least 2 non-empty records in {jsonl_path} for training/eval"
        )

    return texts


def _ensure_min_length(tokens: jax.Array, min_length: int) -> jax.Array:
    """
    Repeats tokens until we have enough positions for batching.

    This keeps the data pipeline simple for very tiny demo corpora.
    """
    if int(tokens.shape[0]) >= min_length:
        return tokens

    repeat_count = int(math.ceil(min_length / int(tokens.shape[0])))
    return jnp.tile(tokens, repeat_count)


def prepare_datasets(
    tokenizer: "PreTrainedTokenizerBase",
    train_texts: list[str],
    val_texts: list[str],
    seq_len: int,
    assistant_only_loss: bool = False,
    user_role_tag: str = "<|user|>",
    assistant_role_tag: str = "<|assistant|>",
) -> tuple[jax.Array, jax.Array, jax.Array, jax.Array]:
    """
    Creates train/val token streams from a list of text samples.

    Works with both plain text (TinyStories) and role-tagged chat text (JSONL).
    When assistant_only_loss=True, also computes a per-token loss mask that
    marks only assistant content as supervised.

    Output format:
    - train_tokens: shape (num_train_tokens,)
    - val_tokens: shape (num_val_tokens,)
    - train_loss_mask: shape (num_train_tokens,)  — all 1.0 when masking is off
    - val_loss_mask: shape (num_val_tokens,)      — all 1.0 when masking is off
    """
    if not train_texts or not val_texts:
        raise ValueError("train_texts and val_texts must both be non-empty")

    def build_stream(texts: list[str]) -> tuple[list[int], list[float]]:
        if not assistant_only_loss:
            joined = "\n\n".join(texts)
            ids = encode_text(tokenizer, joined, add_bos=True, add_eos=True)
            return ids, [1.0] * len(ids)

        _, bos_id, eos_id = _get_special_token_ids(tokenizer)
        sep_ids = _encode_segment(tokenizer, "\n\n")

        all_ids: list[int] = [bos_id]
        all_mask: list[float] = [0.0]

        for text_index, text in enumerate(texts):
            ids, mask = encode_text_with_assistant_mask(
                tokenizer=tokenizer,
                text=text,
                user_role_tag=user_role_tag,
                assistant_role_tag=assistant_role_tag,
            )
            all_ids.extend(ids)
            all_mask.extend(mask)

            if text_index < len(texts) - 1:
                all_ids.extend(sep_ids)
                all_mask.extend([0.0] * len(sep_ids))

        all_ids.append(eos_id)
        all_mask.append(0.0)

        return all_ids, all_mask

    train_ids, train_mask = build_stream(train_texts)
    val_ids, val_mask = build_stream(val_texts)

    train_tokens = jnp.array(train_ids, dtype=jnp.int32)
    val_tokens = jnp.array(val_ids, dtype=jnp.int32)
    train_loss_mask = jnp.array(train_mask, dtype=jnp.float32)
    val_loss_mask = jnp.array(val_mask, dtype=jnp.float32)

    # Ensure both splits are large enough to sample (x, y) windows.
    min_stream_len = seq_len + 2
    train_tokens = _ensure_min_length(train_tokens, min_stream_len)
    val_tokens = _ensure_min_length(val_tokens, min_stream_len)
    train_loss_mask = _ensure_min_length(train_loss_mask, min_stream_len)
    val_loss_mask = _ensure_min_length(val_loss_mask, min_stream_len)

    return train_tokens, val_tokens, train_loss_mask, val_loss_mask


# =============================================================================
# Optimization
# =============================================================================


def cross_entropy_loss(
    logits: jax.Array,
    labels: jax.Array,
    loss_mask: jax.Array | None = None,
) -> jax.Array:
    """Language-model cross-entropy with optional token-level masking."""
    one_hot = jax.nn.one_hot(labels, logits.shape[-1])
    losses = optax.softmax_cross_entropy(logits, one_hot)

    if loss_mask is None:
        return losses.mean()

    mask = loss_mask.astype(losses.dtype)
    masked_sum = jnp.sum(losses * mask)
    denom = jnp.sum(mask)
    return jnp.where(denom > 0, masked_sum / denom, losses.mean())


def create_lr_schedule(
    max_steps: int, warmup_steps: int, peak_lr: float
) -> optax.Schedule:
    """
    Creates a two-phase learning rate schedule:
      Phase 1 (steps 0 .. warmup_steps):        linear ramp  0 → peak_lr
      Phase 2 (steps warmup_steps .. max_steps): cosine decay peak_lr → 0

    This avoids early training instability (warmup) while allowing the
    optimizer to fine-tune at smaller learning rates later (cosine decay).
    """
    warmup = optax.linear_schedule(
        init_value=0.0,
        end_value=peak_lr,
        transition_steps=warmup_steps,
    )
    decay = optax.cosine_decay_schedule(
        init_value=peak_lr,
        decay_steps=max(max_steps - warmup_steps, 1),
    )
    return optax.join_schedules(
        schedules=[warmup, decay],
        boundaries=[warmup_steps],
    )


def _update_all_expert_biases(model: NemotronNanoBlock) -> None:
    """
    Update expert biases across every MoE layer after a training step.

    This is the aux-loss-free load balancing step (Wang et al. 2024, §2.4).
    Each MoE layer stored the top-k indices from its last forward pass in
    `moe.last_topk_indices`. We read those indices here and call
    `update_expert_bias`, which nudges each expert's bias by +/-bias_update_rate
    depending on whether that expert was over- or under-utilized.

    IMPORTANT: Call this AFTER optimizer.update, outside the gradient computation.
    The expert_bias is an nnx.Variable (not nnx.Param) so the optimizer does
    not touch it — it is updated only here.
    """
    for block in model.blocks:
        block.moe.update_expert_bias(block.moe.last_topk_indices[...])


def sample_lm_batch(
    token_stream: jax.Array,
    loss_mask_stream: jax.Array,
    batch_size: int,
    seq_len: int,
    rng_key: jax.Array,
) -> tuple[jax.Array, jax.Array, jax.Array]:
    """
    Samples random contiguous windows for next-token prediction.

    For each sampled window of length (seq_len + 1):
    - x = first seq_len tokens
    - y = next seq_len tokens
    """
    max_start = int(token_stream.shape[0]) - (seq_len + 1)
    if max_start < 0:
        raise ValueError("token_stream is shorter than seq_len + 1")

    starts = jax.random.randint(rng_key, (batch_size,), 0, max_start + 1)

    x_list: list[jax.Array] = []
    y_list: list[jax.Array] = []
    m_list: list[jax.Array] = []
    for start in starts.tolist():
        token_window = token_stream[start : start + seq_len + 1]
        mask_window = loss_mask_stream[start : start + seq_len + 1]
        x_list.append(token_window[:-1])
        y_list.append(token_window[1:])
        m_list.append(mask_window[1:])

    x = jnp.stack(x_list, axis=0)
    y = jnp.stack(y_list, axis=0)
    y_mask = jnp.stack(m_list, axis=0)
    return x, y, y_mask


def train_model(
    model: NemotronNanoBlock,
    optimizer: nnx.Optimizer,
    train_tokens: jax.Array,
    train_loss_mask: jax.Array,
    steps: int,
    batch_size: int,
    seq_len: int,
    rng_key: jax.Array,
    debug_mask_ratio: bool = False,
    debug_mask_every: int = 1,
) -> jax.Array:
    """Runs a tiny training loop and prints readable metrics."""

    if debug_mask_every <= 0:
        raise ValueError("debug_mask_every must be > 0")

    mask_ratios: list[float] = []

    @nnx.jit
    def train_step(
        model: NemotronNanoBlock,
        optimizer: nnx.Optimizer,
        x_batch: jax.Array,
        y_batch: jax.Array,
        y_mask_batch: jax.Array,
    ) -> jax.Array:
        def loss_fn(model: NemotronNanoBlock) -> jax.Array:
            logits_local = model(x_batch)
            return cross_entropy_loss(logits_local, y_batch, loss_mask=y_mask_batch)

        total_loss, grads = nnx.value_and_grad(loss_fn)(model)
        optimizer.update(model, grads)

        # Aux-loss-free load balancing: update expert biases AFTER the gradient step.
        # Uses the top-k indices stored by each SparseMoE during the forward pass.
        _update_all_expert_biases(model)

        return total_loss

    print("\nTraining:")
    for step in range(steps):
        rng_key, batch_key = jax.random.split(rng_key)
        x_batch, y_batch, y_mask_batch = sample_lm_batch(
            train_tokens,
            train_loss_mask,
            batch_size,
            seq_len,
            batch_key,
        )
        total_loss = train_step(model, optimizer, x_batch, y_batch, y_mask_batch)

        print(f"  step {step + 1:>3}/{steps} | ce={float(total_loss):.4f}")
        if debug_mask_ratio:
            supervised_tokens = float(jnp.sum(y_mask_batch))
            total_tokens = int(y_mask_batch.size)
            ratio = supervised_tokens / max(total_tokens, 1)
            mask_ratios.append(ratio)
            
            if ((step + 1) % debug_mask_every == 0):
                print(
                    "    mask-debug "
                    f"supervised={supervised_tokens:.1f}/{total_tokens} "
                    f"ratio={ratio:.4f}"
                )

    if debug_mask_ratio and mask_ratios:
        ratio_min = min(mask_ratios)
        ratio_max = max(mask_ratios)
        ratio_mean = sum(mask_ratios) / len(mask_ratios)
        print(
            "Mask ratio summary "
            f"min={ratio_min:.4f} "
            f"max={ratio_max:.4f} "
            f"mean={ratio_mean:.4f}"
        )

    return rng_key


def evaluate_model(
    model: NemotronNanoBlock,
    val_tokens: jax.Array,
    val_loss_mask: jax.Array,
    batch_size: int,
    seq_len: int,
    eval_batches: int,
    rng_key: jax.Array,
) -> tuple[float, float, jax.Array]:
    """
    Evaluates model on validation batches.

    Returns:
    - mean_ce_loss
    - perplexity = exp(mean_ce_loss)
    - updated rng_key
    """
    ce_losses: list[jax.Array] = []

    for _ in range(eval_batches):
        rng_key, batch_key = jax.random.split(rng_key)
        x_batch, y_batch, y_mask_batch = sample_lm_batch(
            val_tokens,
            val_loss_mask,
            batch_size,
            seq_len,
            batch_key,
        )
        logits = model(x_batch)
        ce_losses.append(cross_entropy_loss(logits, y_batch, loss_mask=y_mask_batch))

    mean_ce = jnp.mean(jnp.stack(ce_losses))
    ppl = jnp.exp(mean_ce)

    return float(mean_ce), float(ppl), rng_key


# =============================================================================
# Chat
# =============================================================================


def pad_or_trim_context(token_ids: list[int], seq_len: int, pad_id: int) -> jax.Array:
    """
    Makes context length exactly `seq_len` so Mamba chunking constraints hold.

    - If context is too long, keep the most recent tokens.
    - If context is too short, left-pad with <pad>.
    """
    if len(token_ids) >= seq_len:
        fixed = token_ids[-seq_len:]
    else:
        pad_count = seq_len - len(token_ids)
        fixed = [pad_id] * pad_count + token_ids

    return jnp.array([fixed], dtype=jnp.int32)


def sample_next_token(
    logits: jax.Array,
    temperature: float,
    rng_key: jax.Array,
) -> int:
    """
    Picks the next token.

    - temperature <= 0: greedy argmax
    - temperature > 0: categorical sampling after temperature scaling
    """
    if temperature <= 0.0:
        return int(jnp.argmax(logits))

    safe_temp = max(temperature, 1e-6)
    scaled = logits / safe_temp
    return int(jax.random.categorical(rng_key, scaled))


def generate_reply(
    model: NemotronNanoBlock,
    tokenizer: "PreTrainedTokenizerBase",
    prompt_text: str,
    seq_len: int,
    max_new_tokens: int,
    temperature: float,
    rng_key: jax.Array,
) -> tuple[str, jax.Array]:
    """Autoregressively generates assistant text from a prompt."""
    pad_id, bos_id, eos_id = _get_special_token_ids(tokenizer)

    context_ids = encode_text(tokenizer, prompt_text, add_bos=True, add_eos=False)
    generated_ids: list[int] = []

    for _ in range(max_new_tokens):
        model_input = pad_or_trim_context(context_ids, seq_len, pad_id)
        logits = model(model_input)

        next_logits = logits[0, -1]

        # Avoid sampling control tokens during response generation.
        # Some tokenizers reuse eos as pad, so guard against masking eos by accident.
        if pad_id != eos_id:
            next_logits = next_logits.at[pad_id].set(-1e9)
        if bos_id != eos_id and bos_id != pad_id:
            next_logits = next_logits.at[bos_id].set(-1e9)

        rng_key, sample_key = jax.random.split(rng_key)
        next_id = sample_next_token(next_logits, temperature, sample_key)

        if next_id == eos_id:
            break

        context_ids.append(next_id)
        generated_ids.append(next_id)

    decoded = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return str(decoded), rng_key


def build_chat_prompt(
    turns: list[tuple[str, str]],
    user_text: str,
    user_role_tag: str,
    assistant_role_tag: str,
) -> str:
    """
    Builds a role-tagged multi-turn prompt from chat history and latest user text.

    Format:
      <|user|>...<|assistant|>...<|user|>...<|assistant|>
    """
    parts: list[str] = []
    for prev_user, prev_assistant in turns:
        parts.append(f"{user_role_tag}\n{prev_user.strip()}\n")
        parts.append(f"{assistant_role_tag}\n{prev_assistant.strip()}\n")

    parts.append(f"{user_role_tag}\n{user_text.strip()}\n")
    parts.append(f"{assistant_role_tag}\n")
    return "".join(parts)


def chat_loop(
    model: NemotronNanoBlock,
    tokenizer: "PreTrainedTokenizerBase",
    seq_len: int,
    temperature: float,
    max_new_tokens: int,
    rng_key: jax.Array,
    user_role_tag: str,
    assistant_role_tag: str,
    history_turns: int,
) -> None:
    """Runs an interactive terminal chatbot with short conversation memory."""
    if history_turns <= 0:
        raise ValueError("history_turns must be > 0")

    print("\nChatbot is ready. Commands: /help, /reset, /exit")
    print(f"Using up to {history_turns} previous turns as prompt context.")

    turns: list[tuple[str, str]] = []

    while True:
        try:
            user_text = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nBot: bye.")
            break

        lower_text = user_text.lower()
        if lower_text in {"quit", "exit", "/exit"}:
            print("Bot: bye.")
            break
        if not user_text:
            continue
        if lower_text == "/help":
            print("Bot: /reset clears memory, /exit ends chat.")
            continue
        if lower_text == "/reset":
            turns.clear()
            print("Bot: conversation memory cleared.")
            continue

        prompt_text = build_chat_prompt(
            turns=turns[-history_turns:],
            user_text=user_text,
            user_role_tag=user_role_tag,
            assistant_role_tag=assistant_role_tag,
        )

        reply, rng_key = generate_reply(
            model=model,
            tokenizer=tokenizer,
            prompt_text=prompt_text,
            seq_len=seq_len,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            rng_key=rng_key,
        )

        reply = reply.strip() or "..."
        turns.append((user_text, reply))
        print(f"Bot: {reply}")


def load_checkpoint_into_model(
    model: NemotronNanoBlock,
    checkpoint_path: str,
) -> int:
    """
    Loads model parameter leaves from a legacy .npz checkpoint.

    Expected checkpoint format:
    - Keys: param_0, param_1, ..., param_N
    - Values: numpy arrays matching current model parameter shapes in order
    """
    path = Path(checkpoint_path)
    if not path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    param_state = nnx.state(model, nnx.Param)
    param_leaves = jax.tree_util.tree_leaves(param_state)
    tree_def = jax.tree_util.tree_structure(param_state)

    with np.load(path, allow_pickle=False) as ckpt:
        keys = list(ckpt.files)
        if not keys:
            raise ValueError(f"Checkpoint is empty: {checkpoint_path}")

        if not all(key.startswith("param_") for key in keys):
            raise ValueError(
                "Checkpoint format mismatch: expected keys like 'param_0', 'param_1', ..."
            )

        try:
            ordered_keys = sorted(keys, key=lambda name: int(name.split("_", 1)[1]))
        except ValueError as exc:
            raise ValueError(
                "Checkpoint keys must follow 'param_<index>' format with integer index"
            ) from exc

        ckpt_leaves = [jnp.asarray(ckpt[key]) for key in ordered_keys]

    if len(ckpt_leaves) != len(param_leaves):
        raise ValueError(
            "Checkpoint parameter count mismatch: "
            f"checkpoint has {len(ckpt_leaves)} leaves, "
            f"model expects {len(param_leaves)} leaves"
        )

    restored_leaves: list[jax.Array] = []
    for leaf_index, (target_leaf, source_leaf) in enumerate(
        zip(param_leaves, ckpt_leaves)
    ):
        target_arr = jnp.asarray(target_leaf)
        if target_arr.shape != source_leaf.shape:
            raise ValueError(
                "Checkpoint shape mismatch at leaf "
                f"{leaf_index}: checkpoint {source_leaf.shape} vs model {target_arr.shape}"
            )

        if source_leaf.dtype != target_arr.dtype:
            source_leaf = source_leaf.astype(target_arr.dtype)

        restored_leaves.append(source_leaf)

    restored_params = jax.tree_util.tree_unflatten(tree_def, restored_leaves)
    nnx.update(model, restored_params)
    return len(restored_leaves)


# =============================================================================
# Main
# =============================================================================


def build_arg_parser() -> argparse.ArgumentParser:
    """CLI arguments kept intentionally small and beginner-friendly."""
    parser = argparse.ArgumentParser(description="Minimal Nemotron train/eval/chat app")
    parser.add_argument("--preset", type=str, default="tiny", help="Nemotron preset")
    parser.add_argument("--steps", type=int, default=80, help="Training steps")
    parser.add_argument(
        "--batch-size", type=int, default=8, help="Train/eval batch size"
    )
    parser.add_argument(
        "--seq-len",
        type=int,
        default=64,
        help="Sequence length (must be divisible by Mamba chunk size)",
    )
    parser.add_argument(
        "--eval-batches", type=int, default=10, help="Validation batches"
    )
    parser.add_argument(
        "--temperature",
        type=float,
        default=0.0,
        help="0 for greedy decoding, >0 for sampling",
    )
    parser.add_argument(
        "--max-new-tokens", type=int, default=80, help="Max tokens per reply"
    )
    parser.add_argument("--seed", type=int, default=0, help="Random seed")
    parser.add_argument(
        "--tinystories-max-stories",
        type=int,
        default=5000,
        help="How many TinyStories stories to load for this run",
    )
    parser.add_argument(
        "--tinystories-train-ratio",
        type=float,
        default=0.9,
        help="Fraction of stories used for training (rest for validation)",
    )
    parser.add_argument(
        "--tinystories-split",
        type=str,
        default="train",
        help="Hugging Face split to read from TinyStories",
    )
    parser.add_argument(
        "--tinystories-cache-dir",
        type=str,
        default=None,
        help="Optional Hugging Face cache directory",
    )
    parser.add_argument(
        "--dataset-format",
        type=str,
        default="tinystories",
        choices=["tinystories", "jsonl"],
        help="Dataset source format: TinyStories streaming or local JSONL files",
    )
    parser.add_argument(
        "--train-jsonl",
        type=str,
        default=None,
        help="Path to train JSONL file (used when --dataset-format=jsonl)",
    )
    parser.add_argument(
        "--val-jsonl",
        type=str,
        default=None,
        help="Path to validation JSONL file (used when --dataset-format=jsonl)",
    )
    parser.add_argument(
        "--jsonl-text-key",
        type=str,
        default="serialized_text",
        help="JSON key to read text from in JSONL records",
    )
    parser.add_argument(
        "--jsonl-max-train-records",
        type=int,
        default=50000,
        help="Max train records to read from train JSONL",
    )
    parser.add_argument(
        "--jsonl-max-val-records",
        type=int,
        default=5000,
        help="Max validation records to read from val JSONL",
    )
    parser.add_argument(
        "--tokenizer-name",
        type=str,
        default="google/byt5-small",
        help="Hugging Face tokenizer name or local path",
    )
    parser.add_argument(
        "--tokenizer-cache-dir",
        type=str,
        default=None,
        help="Optional Hugging Face cache directory for tokenizer files",
    )
    parser.add_argument(
        "--preview-first-story",
        action="store_true",
        help="Print a short preview of the first loaded TinyStories sample",
    )
    parser.add_argument(
        "--skip-chat",
        action="store_true",
        help="Run train+eval only (useful for non-interactive testing)",
    )
    parser.add_argument(
        "--chat-only",
        action="store_true",
        help="Skip train/eval and start chat after loading --checkpoint-path",
    )
    parser.add_argument(
        "--checkpoint-path",
        type=str,
        default=None,
        help="Path to .npz checkpoint with model parameter leaves",
    )
    parser.add_argument(
        "--assistant-only-loss",
        action="store_true",
        help=(
            "Mask loss to assistant content tokens only using role tags in text "
            "(recommended with --dataset-format=jsonl chat data)"
        ),
    )
    parser.add_argument(
        "--user-role-tag",
        type=str,
        default="<|user|>",
        help="Role tag used to mark user turns in serialized chat text",
    )
    parser.add_argument(
        "--assistant-role-tag",
        type=str,
        default="<|assistant|>",
        help="Role tag used to mark assistant turns in serialized chat text",
    )
    parser.add_argument(
        "--debug-mask-ratio",
        action="store_true",
        help="Print supervised-token mask ratio during training",
    )
    parser.add_argument(
        "--debug-mask-every",
        type=int,
        default=1,
        help="Print mask-debug line every N training steps",
    )
    parser.add_argument(
        "--chat-history-turns",
        type=int,
        default=6,
        help="How many previous user-assistant turns to include in chat context",
    )
    return parser


def main() -> None:
    args = build_arg_parser().parse_args()

    if args.steps <= 0:
        raise ValueError("--steps must be > 0")
    if args.batch_size <= 0:
        raise ValueError("--batch-size must be > 0")
    if args.eval_batches <= 0:
        raise ValueError("--eval-batches must be > 0")
    if args.chat_history_turns <= 0:
        raise ValueError("--chat-history-turns must be > 0")
    if args.chat_only and not args.checkpoint_path:
        raise ValueError("--checkpoint-path is required when --chat-only is set")
    if args.chat_only and args.skip_chat:
        raise ValueError("--chat-only and --skip-chat cannot be used together")

    print("Initializing minimal Nemotron app...")

    if args.assistant_only_loss and args.dataset_format != "jsonl":
        print(
            "Note: --assistant-only-loss is most useful with role-tagged chat JSONL data."
        )

    # 1) Load Hugging Face tokenizer.
    tokenizer = load_hf_tokenizer(
        tokenizer_name=args.tokenizer_name,
        cache_dir=args.tokenizer_cache_dir,
    )

    # 2) Build Nemotron config/model.
    config = NemotronConfig.from_preset(args.preset)
    # len(tokenizer) includes any runtime-added special tokens.
    config.vocab_size = len(tokenizer)

    if args.seq_len % config.mamba_chunk_size != 0:
        raise ValueError(
            f"seq_len must be divisible by mamba_chunk_size ({config.mamba_chunk_size})"
        )

    print(
        "Model setup: "
        f"preset={args.preset}, tokenizer={args.tokenizer_name}, "
        f"vocab_size={config.vocab_size}, "
        f"seq_len={args.seq_len}, chunk_size={config.mamba_chunk_size}"
    )

    rngs = nnx.Rngs(args.seed)
    model = NemotronNanoBlock(rngs=rngs, config=config)

    # Chat-only mode: load checkpoint and jump straight to interactive chat.
    if args.chat_only:
        if args.checkpoint_path is None:
            raise ValueError("--checkpoint-path is required for chat-only mode")

        restored_count = load_checkpoint_into_model(
            model=model,
            checkpoint_path=args.checkpoint_path,
        )
        print(
            f"Loaded checkpoint '{args.checkpoint_path}' "
            f"({restored_count} parameter leaves restored)."
        )

        chat_loop(
            model=model,
            tokenizer=tokenizer,
            seq_len=args.seq_len,
            temperature=args.temperature,
            max_new_tokens=args.max_new_tokens,
            rng_key=jax.random.PRNGKey(args.seed + 1),
            user_role_tag=args.user_role_tag,
            assistant_role_tag=args.assistant_role_tag,
            history_turns=args.chat_history_turns,
        )
        return

    # 3) Load text data from selected dataset source.
    if args.dataset_format == "tinystories":
        all_stories = load_tinystories_texts(
            max_stories=args.tinystories_max_stories,
            split=args.tinystories_split,
            cache_dir=args.tinystories_cache_dir,
        )
        train_texts, val_texts = split_train_val_texts(
            stories=all_stories,
            train_ratio=args.tinystories_train_ratio,
        )

        print(
            "Dataset setup: "
            "source=tinystories, "
            f"total_stories={len(all_stories)}, "
            f"train_stories={len(train_texts)}, "
            f"val_stories={len(val_texts)}"
        )

        # Optional preview: helps beginners see real input text before tokenization.
        if args.preview_first_story:
            preview = all_stories[0].replace("\n", " ").strip()
            preview_limit = 240
            if len(preview) > preview_limit:
                preview = preview[:preview_limit] + "..."

            print("\nTinyStories preview (first loaded sample):")
            print(f"  {preview}")
    else:
        if not args.train_jsonl or not args.val_jsonl:
            raise ValueError(
                "--train-jsonl and --val-jsonl are required when --dataset-format=jsonl"
            )

        train_texts = load_jsonl_texts(
            jsonl_path=args.train_jsonl,
            max_records=args.jsonl_max_train_records,
            text_key=args.jsonl_text_key,
        )
        val_texts = load_jsonl_texts(
            jsonl_path=args.val_jsonl,
            max_records=args.jsonl_max_val_records,
            text_key=args.jsonl_text_key,
        )

        print(
            "Dataset setup: "
            "source=jsonl, "
            f"train_records={len(train_texts)}, "
            f"val_records={len(val_texts)}, "
            f"text_key={args.jsonl_text_key}"
        )

        if args.preview_first_story:
            preview = train_texts[0].replace("\n", " ").strip()
            preview_limit = 240
            if len(preview) > preview_limit:
                preview = preview[:preview_limit] + "..."

            print("\nJSONL preview (first loaded sample):")
            print(f"  {preview}")

    # Build optimizer: AdamW with warmup + cosine decay
    max_steps = max(args.steps, 1)
    warmup_steps = max(1, max_steps // 10)
    lr_schedule = create_lr_schedule(
        max_steps=max_steps,
        warmup_steps=warmup_steps,
        peak_lr=3e-4,
    )
    gradient_transformation = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(learning_rate=lr_schedule, weight_decay=0.1),
    )
    optimizer = nnx.Optimizer(model, gradient_transformation, wrt=nnx.Param)

    # Use a separate key stream for data/generation randomness.
    rng_key = jax.random.PRNGKey(args.seed + 1)

    # 4) Prepare token streams from train/validation stories.
    train_tokens, val_tokens, train_loss_mask, val_loss_mask = prepare_datasets(
        tokenizer=tokenizer,
        train_texts=train_texts,
        val_texts=val_texts,
        seq_len=args.seq_len,
        assistant_only_loss=args.assistant_only_loss,
        user_role_tag=args.user_role_tag,
        assistant_role_tag=args.assistant_role_tag,
    )

    # 5) Train.
    rng_key = train_model(
        model=model,
        optimizer=optimizer,
        train_tokens=train_tokens,
        train_loss_mask=train_loss_mask,
        steps=args.steps,
        batch_size=args.batch_size,
        seq_len=args.seq_len,
        rng_key=rng_key,
        debug_mask_ratio=args.debug_mask_ratio,
        debug_mask_every=args.debug_mask_every,
    )

    # 6) Evaluate.
    mean_ce, perplexity, rng_key = evaluate_model(
        model=model,
        val_tokens=val_tokens,
        val_loss_mask=val_loss_mask,
        batch_size=args.batch_size,
        seq_len=args.seq_len,
        eval_batches=args.eval_batches,
        rng_key=rng_key,
    )

    print("\nEvaluation:")
    print(f"  mean CE loss: {mean_ce:.4f}")
    print(f"  perplexity:   {perplexity:.4f}")

    # 7) Chat.
    if not args.skip_chat:
        chat_loop(
            model=model,
            tokenizer=tokenizer,
            seq_len=args.seq_len,
            temperature=args.temperature,
            max_new_tokens=args.max_new_tokens,
            rng_key=rng_key,
            user_role_tag=args.user_role_tag,
            assistant_role_tag=args.assistant_role_tag,
            history_turns=args.chat_history_turns,
        )


In [ ]:
# Experiment configuration
cfg = {
    "preset": "tiny",
    "tokenizer_name": "google/byt5-small",
    "seed": 0,
    "steps": 20,
    "batch_size": 8,
    "seq_len": 64,
    "eval_batches": 5,
    "temperature": 0.0,
    "max_new_tokens": 80,

    # Dataset mode: "tinystories" or "jsonl"
    "dataset_format": "tinystories",

    # TinyStories settings
    "tinystories_max_stories": 2000,
    "tinystories_train_ratio": 0.9,
    "tinystories_split": "train",

    # JSONL settings
    "train_jsonl": "data/oasst2/train.jsonl",
    "val_jsonl": "data/oasst2/val.jsonl",
    "jsonl_text_key": "serialized_text",
    "jsonl_max_train_records": 50000,
    "jsonl_max_val_records": 5000,

    # Assistant-only masking (recommended for chat JSONL)
    "assistant_only_loss": False,
    "user_role_tag": "<|user|>",
    "assistant_role_tag": "<|assistant|>",
    "debug_mask_ratio": False,
    "debug_mask_every": 10,
}

print(cfg)

In [ ]:
# Load data
if cfg["dataset_format"] == "tinystories":
    all_stories = load_tinystories_texts(
        max_stories=cfg["tinystories_max_stories"],
        split=cfg["tinystories_split"],
        cache_dir=None,
    )
    train_texts, val_texts = split_train_val_texts(
        stories=all_stories,
        train_ratio=cfg["tinystories_train_ratio"],
    )
    print(
        f"Loaded TinyStories: total={len(all_stories)} train={len(train_texts)} val={len(val_texts)}"
    )
else:
    train_texts = load_jsonl_texts(
        jsonl_path=cfg["train_jsonl"],
        max_records=cfg["jsonl_max_train_records"],
        text_key=cfg["jsonl_text_key"],
    )
    val_texts = load_jsonl_texts(
        jsonl_path=cfg["val_jsonl"],
        max_records=cfg["jsonl_max_val_records"],
        text_key=cfg["jsonl_text_key"],
    )
    print(
        f"Loaded JSONL: train={len(train_texts)} val={len(val_texts)} key={cfg['jsonl_text_key']}"
    )

In [ ]:
# Build tokenizer, model, optimizer, and token streams
tokenizer = load_hf_tokenizer(cfg["tokenizer_name"], cache_dir=None)

model_cfg = NemotronConfig.from_preset(cfg["preset"])
model_cfg.vocab_size = len(tokenizer)

if cfg["seq_len"] % model_cfg.mamba_chunk_size != 0:
    raise ValueError(
        f"seq_len must be divisible by mamba_chunk_size ({model_cfg.mamba_chunk_size})"
    )

rngs = nnx.Rngs(cfg["seed"])
model = NemotronNanoBlock(rngs=rngs, config=model_cfg)

max_steps = max(cfg["steps"], 1)
warmup_steps = max(1, max_steps // 10)
lr_schedule = create_lr_schedule(max_steps=max_steps, warmup_steps=warmup_steps, peak_lr=3e-4)
tx = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adamw(learning_rate=lr_schedule, weight_decay=0.1),
)
optimizer = nnx.Optimizer(model, tx, wrt=nnx.Param)

train_tokens, val_tokens, train_loss_mask, val_loss_mask = prepare_datasets(
    tokenizer=tokenizer,
    train_texts=train_texts,
    val_texts=val_texts,
    seq_len=cfg["seq_len"],
    assistant_only_loss=cfg["assistant_only_loss"],
    user_role_tag=cfg["user_role_tag"],
    assistant_role_tag=cfg["assistant_role_tag"],
)

rng_key = jax.random.PRNGKey(cfg["seed"] + 1)
print("Prepared token streams.")

In [ ]:
# Train and evaluate
rng_key = train_model(
    model=model,
    optimizer=optimizer,
    train_tokens=train_tokens,
    train_loss_mask=train_loss_mask,
    steps=cfg["steps"],
    batch_size=cfg["batch_size"],
    seq_len=cfg["seq_len"],
    rng_key=rng_key,
    debug_mask_ratio=cfg["debug_mask_ratio"],
    debug_mask_every=cfg["debug_mask_every"],
)

mean_ce, perplexity, rng_key = evaluate_model(
    model=model,
    val_tokens=val_tokens,
    val_loss_mask=val_loss_mask,
    batch_size=cfg["batch_size"],
    seq_len=cfg["seq_len"],
    eval_batches=cfg["eval_batches"],
    rng_key=rng_key,
)

print({"mean_ce": mean_ce, "perplexity": perplexity})

In [ ]:
# Single-shot generation demo (not an interactive chat loop)
prompt = "Once upon a time"
reply, rng_key = generate_reply(
    model=model,
    tokenizer=tokenizer,
    prompt_text=prompt,
    seq_len=cfg["seq_len"],
    max_new_tokens=cfg["max_new_tokens"],
    temperature=cfg["temperature"],
    rng_key=rng_key,
)
print("Prompt:", prompt)
print("Reply:", reply.strip() or "...")

In [ ]:
# Interactive multi-turn chat in notebook
chat_history = []
print("Interactive chat ready. Commands: /reset, /exit")

while True:
    user_text = input("You: ").strip()
    if user_text.lower() in {"/exit", "exit", "quit"}:
        print("Bot: bye.")
        break
    if not user_text:
        continue
    if user_text.lower() == "/reset":
        chat_history.clear()
        print("Bot: conversation memory cleared.")
        continue

    # Keep recent turns to avoid overfilling the fixed context window.
    recent_turns = chat_history[-6:]
    prompt_parts = []
    for prev_user, prev_assistant in recent_turns:
        prompt_parts.append(f"{cfg['user_role_tag']}\n{prev_user}\n")
        prompt_parts.append(f"{cfg['assistant_role_tag']}\n{prev_assistant}\n")
    prompt_parts.append(f"{cfg['user_role_tag']}\n{user_text}\n")
    prompt_parts.append(f"{cfg['assistant_role_tag']}\n")
    prompt_text = "".join(prompt_parts)

    reply, rng_key = generate_reply(
        model=model,
        tokenizer=tokenizer,
        prompt_text=prompt_text,
        seq_len=cfg["seq_len"],
        max_new_tokens=cfg["max_new_tokens"],
        temperature=cfg["temperature"],
        rng_key=rng_key,
    )

    reply = reply.strip() or "..."
    chat_history.append((user_text, reply))
    print("Bot:", reply)